# ⚡ 使用 GitHub 模型的並發代理工作流程（Python）

## 📋 高級並行處理教程

此筆記本展示了使用 Microsoft Agent Framework 的<strong>並發工作流程模式</strong>。您將學習如何構建高性能的並行處理工作流程，其中多個 AI 代理同時執行，顯著提高吞吐量並實現複雜的多線程業務流程。

## 🎯 學習目標

### 🚀 <strong>並發處理基礎</strong>
- <strong>並行代理執行</strong>：同時運行多個代理以獲得最大效率
- <strong>工作流程協調</strong>：在保持數據一致性的同時協調並行操作
- <strong>性能優化</strong>：通過並行處理實現顯著加速
- <strong>資源管理</strong>：高效利用並行操作中的 AI 模型資源

### 🏗️ <strong>高級並發模式</strong>
- **分叉-合併處理**：將工作分配給多個代理並合併結果
- <strong>管道並行</strong>：不同執行階段重疊實現持續吞吐量
- <strong>負載均衡</strong>：均勻分配任務到可用代理資源
- <strong>同步點</strong>：在關鍵工作流程階段協調並行代理

### 🏢 <strong>企業並發應用</strong>
- <strong>高容量文件處理</strong>：同時處理多個文件
- <strong>實時內容分析</strong>：並發分析進入的數據流
- <strong>批處理優化</strong>：最大化大規模操作的吞吐量
- <strong>多模態分析</strong>：並行處理不同類型的內容（文本、圖像、數據）

## ⚙️ 前置條件與設置

### 📦 <strong>所需依賴</strong>

安裝支持並發工作流程的 Agent Framework：

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub 模型配置**

**環境設置（.env 文件）：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**並發處理注意事項：**
- <strong>速率限制</strong>：監控 GitHub 模型 API 對並發請求的速率限制
- <strong>資源使用</strong>：考慮多個並行代理的記憶體和 CPU 使用情況
- <strong>錯誤處理</strong>：為並行操作實施強健的錯誤恢復機制

### 🏗️ <strong>並發工作流程架構</strong>

```mermaid
graph TD
    A[工作流程開始] --> B[並行執行]
    B --> C[代理池 1]
    B --> D[代理池 2]
    B --> E[代理池 3]
    C --> F[結果匯總]
    D --> F
    E --> F
    F --> G[最終輸出]
    
    H[GitHub 模型 API] --> C
    H --> D
    H --> E
```

**主要優點：**
- **⚡ 性能**：通過並行執行顯著加速
- **📈 可擴展性**：處理增長的工作負載而不成比例增加時間
- **🔄 效率**：更好地利用可用計算資源
- **🎯 吞吐量**：在相同時間內處理更多工作

## 🎨 <strong>並發工作流程設計模式</strong>

### 🔍 <strong>研究與分析管道</strong>
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 <strong>數據處理工作流程</strong>
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 <strong>內容創作管道</strong>
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 <strong>多階段處理</strong>
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 <strong>企業性能收益</strong>

### ⚡ <strong>吞吐量優化</strong>
- <strong>並行執行</strong>：多個代理同時工作
- <strong>資源利用</strong>：最大化可用 AI 模型容量效率
- <strong>時間縮短</strong>：顯著減少總處理時間
- <strong>可擴展架構</strong>：根據需要輕鬆添加更多並行代理

### 🛡️ <strong>可靠性與韌性</strong>
- <strong>容錯</strong>：個別代理失敗不會中斷整個工作流程
- <strong>錯誤隔離</strong>：一個並發分支的問題不影響其他分支
- <strong>漸進降級</strong>：即使代理容量減少系統仍繼續運行
- <strong>恢復機制</strong>：自動重試和錯誤處理以應對失敗操作

### 📊 <strong>監控與可觀察性</strong>
- <strong>並行執行跟蹤</strong>：監控所有並行操作的進度
- <strong>性能指標</strong>：衡量加速與效率提升
- <strong>資源使用分析</strong>：優化並行代理分配
- <strong>瓶頸識別</strong>：發現並解決性能瓶頸

讓我們一起構建高性能的並發 AI 工作流程！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
